In [0]:
df = spark.read.table("workspace.bronze.crm_sales_details")

In [0]:
df.display()

### Data Transformations

In [0]:
## trim whitespace 
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col
from pyspark.sql.types import StringType

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df1 = df.withColumn(field.name, trim(col(field.name)))

## rename columns 
mapping ={
    'sls_ord_num':'sales_order_number', 
    'sls_prd_key':'sales_product_key',
    'sls_cust_id':'sales_customer_id',
    'sls_order_dt': 'sales_order_date',
    'sls_ship_dt': 'sales_ship_date',
    'sls_due_dt':'sales_due_date',
    'sls_sales':'sales_amount',
    'sls_quantity':'sales_quantity',
    'sls_price':'sales_price'
}
df2 = df1.withColumnsRenamed(mapping)
df2.display()

In [0]:
## change the data type of sales_order_date, sales_ship_date, sales_due_date to date
from pyspark.sql.functions import try_to_date
df3 = df2.withColumn('sales_order_date', try_to_date(df2['sales_order_date'].cast('string'), 'yyyyMMdd'))
df3 = df3.withColumn('sales_ship_date', try_to_date(df3['sales_ship_date'].cast('string'), 'yyyyMMdd'))
df3 = df3.withColumn('sales_due_date', try_to_date(df3['sales_due_date'].cast('string'), 'yyyyMMdd'))
df3.display()

In [0]:
from pyspark.sql.functions import concat_ws

df4 = df3.withColumn("order_product_unique", concat_ws("_", df3.sales_order_number, df3.sales_product_key))
df_duplicates = df4.groupBy("order_product_unique").count().filter("count > 1")
display(df_duplicates)

### write to silver table 

In [0]:
df3.write.mode("overwrite").format('delta').saveAsTable("workspace.silver.crm_sales_info_cleaned")